In [ ]:
#| default_exp gate

In [ ]:
#| export
from __future__ import annotations

import os
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

from fastermodels.card import _is_count, check_card

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

The gate reads the artifact that was **produced**, not the intentions of the script that produced it: it
reloads the weights in a fresh interpreter, reads the exported ONNX back, and reads the card back.

| # | Condition | Passes when |
|---|---|---|
| 0 | license | the license has an id and the name that validated it |
| 1 | fresh-interpreter reload | a scrubbed subprocess reloads the artifact to the published hash |
| 2 | parity | every same-precision arm agrees on at least 99.5 % of the images, and every batch-invariance arm differs by at most 1e-5 |
| 3 | accuracy delta | the low end of the paired interval is above the floor |
| 4 | exported file | the ONNX the manifest claims is there, with opset 17, a dynamic batch and its Q/DQ pairs |
| 5 | head and widths | the head has the expected number of classes and the widths are recorded |
| 6 | size, memory, MACs and latency | every row carries size, memory and MACs as non-negative ints, and there are latency rows or the literal `non mesurée` |
| 7 | card | the README is there and `check_card` flags nothing |
| 8 | clean-machine reload | a clean machine reloaded the artifact and got the same result |
| 9 | on-target claim | latency rows come with a proof measured on the target |

A number that is missing is not a pass: condition 3 fails when there is no `lo` to read, and condition 6
fails naming the criterion a row does not carry.

In [ ]:
#| export
@dataclass(slots=True)
class GateRow:
    "One publication condition and what it found"
    condition: int
    name: str
    passed: bool
    evidence: str

    def as_dict(self) -> dict: return asdict(self)


def _reload_hash(artifact_dir, python, pythonpath):
    "State hash of the artifact reloaded by a fresh interpreter with a scrubbed environment"
    code = ("from fastermodels import FasterModel, state_hash\n"
            f"print(state_hash(FasterModel.from_pretrained({str(artifact_dir)!r})))")
    run = subprocess.run([python, '-c', code], capture_output=True, text=True,
                         env={'PATH': os.environ.get('PATH', ''), 'PYTHONPATH': pythonpath or ''})
    if run.returncode != 0 or not run.stdout.strip():
        return None, (run.stderr.strip() or 'no output').splitlines()[-1]
    return run.stdout.strip().splitlines()[-1], ''


def _onnx_conditions(path, claimed):
    "Post-conditions read back from the exported ONNX itself, never from what the manifest claims about it"
    if not claimed:
        return (False, 'model.onnx is there but the manifest has no files.onnx entry') if path.exists() \
            else (True, 'no model.onnx in the artifact directory')
    if not path.exists(): return False, 'model.onnx claimed in the manifest but missing'
    try:
        import onnx
    except ImportError:
        return False, 'install onnx to verify model.onnx'
    graph = onnx.load(str(path))
    opset = max(i.version for i in graph.opset_import if i.domain in ('', 'ai.onnx'))
    dynamic = graph.graph.input[0].type.tensor_type.shape.dim[0].HasField('dim_param')
    n_q = sum(n.op_type == 'QuantizeLinear' for n in graph.graph.node)
    n_dq = sum(n.op_type == 'DequantizeLinear' for n in graph.graph.node)
    qdq = (claimed.get('n_q') or 0) > 0 or (claimed.get('n_dq') or 0) > 0   # the manifest claims Q/DQ, n_q and n_dq come from the file
    passed = opset == 17 and dynamic and (not qdq or (n_q > 0 and n_dq > 0))
    return passed, f"opset={opset} dynamic_batch={dynamic} n_q={n_q} n_dq={n_dq}"


def run_gate(
    artifact_dir: str | Path,       # directory holding config.json, model.safetensors and README.md
    manifest: dict,                 # what the producer claims about the artifact
    *,
    python: str | None = None,      # interpreter of the fresh-interpreter reload (default: this one)
    pythonpath: str | None = None,  # the only PYTHONPATH that reload is given
) -> list[GateRow]:
    "Run the ten publication conditions on a local artifact directory"
    d, rows = Path(artifact_dir), []
    def add(condition, name, passed, evidence): rows.append(GateRow(condition, name, bool(passed), evidence))

    lic = manifest.get('license') or {}
    add(0, 'license', lic.get('id') and lic.get('validated_by'),
        f"id={lic.get('id')!r} validated by {lic.get('validated_by')!r}")

    claimed = (manifest.get('hashes') or {}).get('safetensors')
    got, err = _reload_hash(d, python or sys.executable, pythonpath)
    add(1, 'fresh-interpreter reload', got is not None and got == claimed,
        f"reloaded {got} vs manifest {claimed}" if got else f"reload failed: {err}")

    parity = manifest.get('parity') or []
    same = [p for p in parity if p.get('kind') == 'same-precision']
    batch = [p for p in parity if p.get('kind') == 'batch-invariance']
    add(2, 'parity', same and all((p.get('agreement') or 0) >= 0.995 for p in same)
        and all(p.get('max_abs_diff') is not None and p['max_abs_diff'] <= 1e-5 for p in batch),
        '; '.join([f"{p.get('arms')} {p.get('agreement')}" for p in same]
                  + [f"{p.get('arms')} max_abs_diff={p.get('max_abs_diff')}" for p in batch])
        or 'no same-precision parity arm')

    delta = manifest.get('delta') or {}
    lo, floor = delta.get('lo'), delta.get('floor')
    add(3, 'accuracy delta above floor', lo is not None and floor is not None and lo > floor,
        f"delta={delta.get('delta')} lo={lo} hi={delta.get('hi')} floor={floor}"
        + ('' if lo is not None else ' — no interval, so no verdict'))

    add(4, 'exported file', *_onnx_conditions(d / 'model.onnx', (manifest.get('files') or {}).get('onnx')))

    head, widths = manifest.get('head') or {}, manifest.get('widths') or {}
    add(5, 'head and widths', head.get('expected') is not None and head.get('num_classes') == head.get('expected') and widths,
        f"num_classes={head.get('num_classes')} expected={head.get('expected')} widths={len(widths)} layers")

    latency, measured = manifest.get('latency_rows'), manifest.get('rows') or []
    missing = [f"row {i} has no {f}" for i, r in enumerate(measured) for f in ('bytes', 'peak_activation_bytes', 'macs')
               if not (_is_count(r.get(f)) and r[f] >= 0)]
    add(6, 'size, memory, MACs and latency',
        measured and not missing and ((isinstance(latency, list) and len(latency) > 0) or latency == 'non mesurée'),
        '; '.join(missing) or (f"{len(measured)} rows measured, latency_rows="
                               + (f"{len(latency)} rows" if isinstance(latency, list) else f"{latency!r}")))

    card = d / 'README.md'
    flagged = check_card(card.read_text()) if card.exists() else None
    add(7, 'card', flagged == [], 'README.md missing' if flagged is None else (f"flagged: {flagged}" if flagged else 'nothing flagged'))

    clean = manifest.get('clean_reload') or {}
    add(8, 'clean-machine reload', clean.get('passed'), f"clean reload {clean}" if clean else 'not run')

    add(9, 'on-target claim', manifest.get('proof') or latency == 'non mesurée',
        'proof measured on the target' if manifest.get('proof') else
        ('no on-target claim, latency non mesurée' if latency == 'non mesurée' else 'latency rows without a proof on the target'))
    return rows


def gate_passed(
    rows: list[GateRow],  # what `run_gate` returned
) -> bool:
    "True when every condition passed"
    return all(r.passed for r in rows)

In [ ]:
show_doc(run_gate)

In [ ]:
show_doc(GateRow)

In [ ]:
show_doc(gate_passed)

---

## Usage

```python
from fastermodels import run_gate, gate_passed

rows = run_gate('artifact', manifest, python=sys.executable, pythonpath='/path/to/fastermodels')
for r in rows: print(f"{r.condition} {r.name:26} {'pass' if r.passed else 'FAIL'}  {r.evidence}")
gate_passed(rows)
```

```
0 license                    pass  id='bsd-3-clause' validated by 'nathan'
1 fresh-interpreter reload   pass  reloaded 6f3c... vs manifest 6f3c...
2 parity                     pass  ['in-memory', 'safetensors'] 1.0; ['batch 1', 'batch 32'] max_abs_diff=1e-07
3 accuracy delta above floor pass  delta=-0.51 lo=-1.2 hi=0.2 floor=-2.0
6 size, memory, MACs and ... pass  1 rows measured, latency_rows='non mesurée'
...
```

The reload of condition 1 runs with `PYTHONPATH` set to that one path and nothing else, so it fails if the
artifact only reloads thanks to something in the producer's environment.

---

## See Also

- [Model](00_model.html) - `state_hash`, the digest condition 1 compares
- [Card](02_card.html) - `check_card`, which condition 7 runs
- [Eval](01_eval.html) - the paired interval condition 3 reads

Tests live in `nbs/tests/test_gate.ipynb`.